<a href="https://www.kaggle.com/code/aabdollahii/1-2-analysis-graph?scriptVersionId=339536512" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Watching Graph

In [1]:
# Core libraries for file handling and data inspection
from pathlib import Path
import json
import os
import random

# Data libraries
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

# Root directory of the saved Kaggle notebook output
ROOT_DIR = Path("/kaggle/input/notebooks/aabdollahii/1-1-wikimedia-second-try")

# Graph output directory
GRAPH_DIR = ROOT_DIR / "fawiki_graph"

# Subdirectories
PAGES_DIR = GRAPH_DIR / "pages"
EDGES_DIR = GRAPH_DIR / "edges"

# Metadata files
METADATA_PATH = GRAPH_DIR / "metadata.json"
FINAL_REPORT_PATH = GRAPH_DIR / "final_report.json"

print("ROOT_DIR exists:", ROOT_DIR.exists())
print("GRAPH_DIR exists:", GRAPH_DIR.exists())
print("PAGES_DIR exists:", PAGES_DIR.exists())
print("EDGES_DIR exists:", EDGES_DIR.exists())
print("METADATA_PATH exists:", METADATA_PATH.exists())
print("FINAL_REPORT_PATH exists:", FINAL_REPORT_PATH.exists())


ROOT_DIR exists: True
GRAPH_DIR exists: True
PAGES_DIR exists: True
EDGES_DIR exists: True
METADATA_PATH exists: True
FINAL_REPORT_PATH exists: True


In [2]:
# Read metadata and final report for quick inspection
with open(METADATA_PATH, "r", encoding="utf-8") as file:
    metadata = json.load(file)

with open(FINAL_REPORT_PATH, "r", encoding="utf-8") as file:
    final_report = json.load(file)

print("Metadata:")
print(json.dumps(metadata, ensure_ascii=False, indent=2))

print("\n" + "=" * 80 + "\n")

print("Final report:")
print(json.dumps(final_report, ensure_ascii=False, indent=2))


Metadata:
{
  "status": "processed",
  "dump_path": "/kaggle/working/fawiki-latest-pages-articles-multistream.xml.bz2",
  "output_dir": "/kaggle/working/fawiki_graph",
  "pages_dir": "/kaggle/working/fawiki_graph/pages",
  "edges_dir": "/kaggle/working/fawiki_graph/edges",
  "batch_size": 5000,
  "number_of_batches": 612,
  "total_pages": 3056694,
  "total_edges_after_batch_level_deduplication": 25698688,
  "total_redirects": 1977522,
  "failed_pages": 0,
  "elapsed_seconds": 6131.842935800552,
  "elapsed_minutes": 102.19738226334253,
  "relations": [
    "links_to",
    "belongs_to_category",
    "redirects_to"
  ],
  "note": "This graph is a Persian Wikipedia page-link graph. It contains article nodes, redirect nodes, category edges, internal page links, and redirect edges."
}


Final report:
{
  "status": "completed",
  "output_dir": "/kaggle/working/fawiki_graph",
  "pages_dir": "/kaggle/working/fawiki_graph/pages",
  "edges_dir": "/kaggle/working/fawiki_graph/edges",
  "num_page_f

In [3]:
# Collect all page and edge parquet files
page_files = sorted(PAGES_DIR.glob("*.parquet"))
edge_files = sorted(EDGES_DIR.glob("*.parquet"))

print("Number of page parquet files:", len(page_files))
print("Number of edge parquet files:", len(edge_files))

print("\nFirst 5 page files:")
for path in page_files[:5]:
    print(path)

print("\nFirst 5 edge files:")
for path in edge_files[:5]:
    print(path)


Number of page parquet files: 612
Number of edge parquet files: 612

First 5 page files:
/kaggle/input/notebooks/aabdollahii/1-1-wikimedia-second-try/fawiki_graph/pages/pages_00000.parquet
/kaggle/input/notebooks/aabdollahii/1-1-wikimedia-second-try/fawiki_graph/pages/pages_00001.parquet
/kaggle/input/notebooks/aabdollahii/1-1-wikimedia-second-try/fawiki_graph/pages/pages_00002.parquet
/kaggle/input/notebooks/aabdollahii/1-1-wikimedia-second-try/fawiki_graph/pages/pages_00003.parquet
/kaggle/input/notebooks/aabdollahii/1-1-wikimedia-second-try/fawiki_graph/pages/pages_00004.parquet

First 5 edge files:
/kaggle/input/notebooks/aabdollahii/1-1-wikimedia-second-try/fawiki_graph/edges/edges_00000.parquet
/kaggle/input/notebooks/aabdollahii/1-1-wikimedia-second-try/fawiki_graph/edges/edges_00001.parquet
/kaggle/input/notebooks/aabdollahii/1-1-wikimedia-second-try/fawiki_graph/edges/edges_00002.parquet
/kaggle/input/notebooks/aabdollahii/1-1-wikimedia-second-try/fawiki_graph/edges/edges_0000

In [4]:
# Read one sample page parquet file and one sample edge parquet file
sample_pages_df = pd.read_parquet(page_files[0])
sample_edges_df = pd.read_parquet(edge_files[0])

print("Sample pages shape:", sample_pages_df.shape)
print("Sample edges shape:", sample_edges_df.shape)

display(sample_pages_df.head(10))
display(sample_edges_df.head(10))


Sample pages shape: (5000, 8)
Sample edges shape: (339161, 3)


,page_id,title,namespace,is_redirect,redirect_target,text,num_links,num_categories
0,2,صفحهٔ اصلی,0,False,None,15px|alt=|پیوند= مقاله‌های برگزیده – مقالهٔ ام...,10,0
1,594,ویکی‌پدیا,0,False,None,ویکی‌پدیا ، یک دانشنامه برخط چندزبانه بر پایهٔ...,189,21
2,613,سالنامه,0,False,None,بندانگشتی|جلد یک سالنامه\nسالنامه‌ها یا وقایع‌...,20,4
3,619,اطلاعات,0,False,None,بندانگشتی|منابع اِطّلاعات، اَزدایش\nاِطّلاعات،...,23,8
4,637,محتوای آزاد,0,False,None,محتوای آزاد یا اطلاعات آزاد یا محتوای باز هر ا...,55,6
5,643,ویکی,0,False,None,بندانگشتی|چپ|مصاحبه با وارد کانینگهام، خالق وی...,47,7
6,645,نرم‌افزارهای مشارکت‌گرا,0,False,None,بندانگشتی|تصویری از یک نرم افزار مشارکت گرا\nن...,1,8
7,646,سعدی,0,False,None,ابومحمّدْ مُشرف‌الدینْ مُصلِح بن عبدالله بن مش...,595,25
8,649,جامی,0,False,None,نورالدّین عبد الرّحمن بن احمد بن محمد جامی (۲۳...,106,19
9,650,آیزاک نیوتن,0,False,None,سِر آیزاک نیوتُن (؛ ۲۵ دسامبر ۱۶۴۲ – ۲۰ مارس ۱...,154,40


,source,relation,target
0,صفحهٔ اصلی,links_to,"Y ""(میلادی)"""
1,صفحهٔ اصلی,links_to,j F
2,صفحهٔ اصلی,links_to,xiY (خورشیدی)
3,صفحهٔ اصلی,links_to,xij xiF
4,صفحهٔ اصلی,links_to,xmY (قمری)
5,صفحهٔ اصلی,links_to,xmj xmF
6,صفحهٔ اصلی,links_to,تقویم میلادی
7,صفحهٔ اصلی,links_to,ساعت هماهنگ جهانی
8,صفحهٔ اصلی,links_to,گاه‌شماری هجری خورشیدی
9,صفحهٔ اصلی,links_to,گاه‌شماری هجری قمری قراردادی


In [5]:
# Show page and edge schemas
print("Page columns:")
print(sample_pages_df.columns.tolist())

print("\nEdge columns:")
print(sample_edges_df.columns.tolist())

print("\nPage dtypes:")
print(sample_pages_df.dtypes)

print("\nEdge dtypes:")
print(sample_edges_df.dtypes)


Page columns:
['page_id', 'title', 'namespace', 'is_redirect', 'redirect_target', 'text', 'num_links', 'num_categories']

Edge columns:
['source', 'relation', 'target']

Page dtypes:
page_id             int64
title              object
namespace           int64
is_redirect          bool
redirect_target    object
text               object
num_links           int64
num_categories      int64
dtype: object

Edge dtypes:
source      object
relation    object
target      object
dtype: object


In [6]:
# Count total rows across all parquet files without loading everything into one dataframe
def count_rows_in_parquet_files(parquet_files):
    total_rows = 0

    for file_path in tqdm(parquet_files):
        parquet_file = pq.ParquetFile(file_path)
        total_rows += parquet_file.metadata.num_rows

    return total_rows

total_page_rows = count_rows_in_parquet_files(page_files)
total_edge_rows = count_rows_in_parquet_files(edge_files)

print("Total page rows from parquet files:", total_page_rows)
print("Total edge rows from parquet files:", total_edge_rows)

print("\nMetadata total_pages:", metadata["total_pages"])
print("Metadata total_edges_after_batch_level_deduplication:", metadata["total_edges_after_batch_level_deduplication"])


  0%|          | 0/612 [00:00<?, ?it/s]

  0%|          | 0/612 [00:00<?, ?it/s]

Total page rows from parquet files: 3056694
Total edge rows from parquet files: 25698688

Metadata total_pages: 3056694
Metadata total_edges_after_batch_level_deduplication: 25698688


In [7]:
# Read a few random parquet files to make sure the structure is consistent everywhere
random_page_files = random.sample(page_files, min(3, len(page_files)))
random_edge_files = random.sample(edge_files, min(3, len(edge_files)))

print("Random page files:")
for file_path in random_page_files:
    df = pd.read_parquet(file_path)
    print(file_path.name, "shape =", df.shape)

print("\nRandom edge files:")
for file_path in random_edge_files:
    df = pd.read_parquet(file_path)
    print(file_path.name, "shape =", df.shape)


Random page files:
pages_00420.parquet shape = (5000, 8)
pages_00437.parquet shape = (5000, 8)
pages_00170.parquet shape = (5000, 8)

Random edge files:
edges_00391.parquet shape = (41682, 3)
edges_00408.parquet shape = (49891, 3)
edges_00130.parquet shape = (9049, 3)


In [8]:
# Aggregate page-level statistics
total_redirect_rows = 0
total_non_redirect_rows = 0
pages_with_empty_text = 0

for file_path in tqdm(page_files):
    df = pd.read_parquet(
        file_path,
        columns=["is_redirect", "text", "num_links", "num_categories"]
    )

    total_redirect_rows += int(df["is_redirect"].sum())
    total_non_redirect_rows += int((~df["is_redirect"]).sum())
    pages_with_empty_text += int((df["text"].fillna("").str.len() == 0).sum())

print("Total redirect pages:", total_redirect_rows)
print("Total non-redirect pages:", total_non_redirect_rows)
print("Pages with empty text:", pages_with_empty_text)


  0%|          | 0/612 [00:00<?, ?it/s]

Total redirect pages: 1977522
Total non-redirect pages: 1079172
Pages with empty text: 1977531


In [9]:
# Validate edge integrity
bad_source_count = 0
bad_target_count = 0
self_loop_count = 0

for file_path in tqdm(edge_files):
    df = pd.read_parquet(file_path, columns=["source", "target", "relation"])

    bad_source_count += int(df["source"].fillna("").str.strip().eq("").sum())
    bad_target_count += int(df["target"].fillna("").str.strip().eq("").sum())
    self_loop_count += int((df["source"] == df["target"]).sum())

print("Edges with empty source:", bad_source_count)
print("Edges with empty target:", bad_target_count)
print("Edges with self-loop:", self_loop_count)


  0%|          | 0/612 [00:00<?, ?it/s]

Edges with empty source: 0
Edges with empty target: 0
Edges with self-loop: 48575


In [10]:
# Search a page by title across all page parquet files
def find_page_by_title(title, page_files):
    matches = []

    for file_path in tqdm(page_files):
        df = pd.read_parquet(
            file_path,
            columns=[
                "page_id",
                "title",
                "namespace",
                "is_redirect",
                "redirect_target",
                "text",
                "num_links",
                "num_categories"
            ]
        )

        result = df[df["title"] == title]

        if len(result) > 0:
            matches.append(result)

    if matches:
        return pd.concat(matches, ignore_index=True)

    return pd.DataFrame(columns=[
        "page_id",
        "title",
        "namespace",
        "is_redirect",
        "redirect_target",
        "text",
        "num_links",
        "num_categories"
    ])
iran_page = find_page_by_title("ایران", page_files)
display(iran_page)


  0%|          | 0/612 [00:00<?, ?it/s]

,page_id,title,namespace,is_redirect,redirect_target,text,num_links,num_categories
0,163930,ایران,0,False,None,ایران با نام رسمیِ جمهوری اسلامی ایران، کشوری ...,1297,19


In [11]:
# Search edges connected to a specific title
def find_edges_for_title(title, edge_files, limit_per_file=None):
    matched_frames = []

    for file_path in tqdm(edge_files):
        df = pd.read_parquet(file_path, columns=["source", "relation", "target"])

        result = df[(df["source"] == title) | (df["target"] == title)]

        if limit_per_file is not None:
            result = result.head(limit_per_file)

        if len(result) > 0:
            matched_frames.append(result)

    if matched_frames:
        return pd.concat(matched_frames, ignore_index=True)

    return pd.DataFrame(columns=["source", "relation", "target"])


iran_edges = find_edges_for_title("ایران", edge_files)
print("Number of edges connected to Iran:", len(iran_edges))
display(iran_edges.head(50))


  0%|          | 0/612 [00:00<?, ?it/s]

Number of edges connected to Iran: 98296


,source,relation,target
0,ویکی‌پدیا,links_to,ایران
1,سعدی,links_to,ایران
2,گاه‌شماری میلادی,links_to,ایران
3,عماد خراسانی,links_to,ایران
4,۱۴ اسفند,links_to,ایران
5,محمد مصدق,links_to,ایران
6,اریک کلپتون,links_to,ایران
7,ئاشتی,links_to,ایران
8,آشپزی,links_to,ایران
9,تهران,links_to,ایران


In [12]:
# Split edges into outgoing and incoming groups
def split_outgoing_incoming(title, edges_df):
    outgoing_df = edges_df[edges_df["source"] == title].copy()
    incoming_df = edges_df[edges_df["target"] == title].copy()
    return outgoing_df, incoming_df

iran_outgoing, iran_incoming = split_outgoing_incoming("ایران", iran_edges)

print("Outgoing edges:", len(iran_outgoing))
print("Incoming edges:", len(iran_incoming))

display(iran_outgoing.head(20))
display(iran_incoming.head(20))


Outgoing edges: 1316
Incoming edges: 96981


,source,relation,target
5048,ایران,links_to,Al Jazeera
5049,ایران,links_to,Ali Khamenei
5050,ایران,links_to,Economist Intelligence Unit
5051,ایران,links_to,Encyclopædia Britannica
5052,ایران,links_to,Freedom House
5053,ایران,links_to,International Monetary Fund
5054,ایران,links_to,Reuters
5055,ایران,links_to,Supreme Leader of Iran
5056,ایران,links_to,The Economist
5057,ایران,links_to,The New York Times


,source,relation,target
0,ویکی‌پدیا,links_to,ایران
1,سعدی,links_to,ایران
2,گاه‌شماری میلادی,links_to,ایران
3,عماد خراسانی,links_to,ایران
4,۱۴ اسفند,links_to,ایران
5,محمد مصدق,links_to,ایران
6,اریک کلپتون,links_to,ایران
7,ئاشتی,links_to,ایران
8,آشپزی,links_to,ایران
9,تهران,links_to,ایران


In [13]:
# Lightweight duplicate check on a sample of edge files
sampled_edge_files = random.sample(edge_files, min(20, len(edge_files)))

duplicate_edge_count_sample = 0
sample_edge_counts = {}

for file_path in tqdm(sampled_edge_files):
    df = pd.read_parquet(file_path, columns=["source", "relation", "target"])

    for row in df.itertuples(index=False):
        key = (row.source, row.relation, row.target)

        if key in sample_edge_counts:
            sample_edge_counts[key] += 1
            duplicate_edge_count_sample += 1
        else:
            sample_edge_counts[key] = 1

print("Duplicate edges in sampled files:", duplicate_edge_count_sample)
print("Unique sampled edges:", len(sample_edge_counts))


  0%|          | 0/20 [00:00<?, ?it/s]

Duplicate edges in sampled files: 0
Unique sampled edges: 697844


In [14]:
# Build a sampled page-title set for a lightweight sanity check
sampled_page_files = random.sample(page_files, min(30, len(page_files)))

sample_page_titles = set()

for file_path in tqdm(sampled_page_files):
    df = pd.read_parquet(file_path, columns=["title"])
    sample_page_titles.update(df["title"].dropna().astype(str).tolist())

print("Sampled page title count:", len(sample_page_titles))


  0%|          | 0/30 [00:00<?, ?it/s]

Sampled page title count: 150000


In [15]:
# Check how many edge targets appear in the sampled page-title set
sampled_edge_files = random.sample(edge_files, min(20, len(edge_files)))

matched_targets = 0
unmatched_targets = 0

for file_path in tqdm(sampled_edge_files):
    df = pd.read_parquet(file_path, columns=["target"])

    for target in df["target"].dropna().astype(str):
        if target in sample_page_titles:
            matched_targets += 1
        else:
            unmatched_targets += 1

print("Matched targets in sample title set:", matched_targets)
print("Unmatched targets in sample title set:", unmatched_targets)


  0%|          | 0/20 [00:00<?, ?it/s]

Matched targets in sample title set: 23550
Unmatched targets in sample title set: 914635
